# Modeling 1: 10 training reviews, four classifiers

This is the **California Cabernet Sauvignon (red) versus Chardonnay (white)** stage. All four classifiers use `description` only. The training reviews come from saved `train.parquet`; every eligible review from saved `validation.parquet` is used for validation. `test.parquet` is never loaded.

The 10-, 100-, and 1,000-review notebooks use the same deterministic 1,000-review ordering, so the smaller training sets are nested inside the larger ones. The validation set and TF-IDF settings are identical across them. DistilBERT uses the same rows, but pretrained language features and a different text representation. The original 78% and 93% results sampled before the saved split and are not directly comparable to these new scores.

Run the classical models locally or in Colab. The DistilBERT section needs PyTorch, Transformers, and Accelerate; a Colab GPU is recommended. In Colab, upload `train.parquet` and `validation.parquet` together when prompted. The BERT section writes temporary checkpoints. It uses a PyTorch dataset; the setup cell also upgrades Hugging Face `datasets` because Transformers may import an installed copy internally.

## 1. Load the saved partitions

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
LABELS = ['red', 'white']
if not all(Path(name).exists() for name in ['train.parquet', 'validation.parquet']):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place train.parquet and validation.parquet beside this notebook.') from exc
    print('Upload train.parquet and validation.parquet together.')
    files.upload()
assert all(Path(name).exists() for name in ['train.parquet', 'validation.parquet'])
saved_train = pd.read_parquet('train.parquet')
saved_validation = pd.read_parquet('validation.parquet')
assert saved_train.index.intersection(saved_validation.index).empty
print(f'Saved partitions: {len(saved_train):,} train; {len(saved_validation):,} validation')

Saved partitions: 101,506 train; 33,835 validation


## 2. Make the nested training sample and use the full in-scope validation partition

"Full validation" here means **all California Cabernet Sauvignon and Chardonnay reviews in `validation.parquet`**. Reviews of other varieties are a different target population and belong in a later expansion stage.

In [2]:
N_TRAIN = 10
MODEL_NUMBER = 1
expected_colors = {'Cabernet Sauvignon': 'red', 'Chardonnay': 'white'}
def narrow_pool(frame):
    pool = frame.loc[frame['province'].eq('California') & frame['variety'].isin(expected_colors)].dropna(subset=['description', 'color']).copy()
    pool['description_key'] = pool['description'].str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
    pool = pool.loc[pool['description_key'].ne('')]
    pool = pool.loc[pool['color'].eq(pool['variety'].map(expected_colors))]
    conflicts = pool.groupby('description_key')['color'].nunique()
    pool = pool.loc[~pool['description_key'].isin(conflicts[conflicts > 1].index)]
    return pool.drop_duplicates('description_key')

train_pool = narrow_pool(saved_train)
validation_all = narrow_pool(saved_validation)
assert set(train_pool['description_key']).isdisjoint(validation_all['description_key'])
# Sample once at the largest size. Taking prefixes makes 10 ⊂ 100 ⊂ 1,000.
ordered_train = {
    color: train_pool.loc[train_pool['color'].eq(color)].sample(n=500, random_state=SEED)
    for color in LABELS
}
train_sample = pd.concat([ordered_train[color].iloc[:N_TRAIN // 2] for color in LABELS]).sample(frac=1, random_state=SEED)
assert len(train_sample) == N_TRAIN
assert set(train_sample['description_key']).isdisjoint(validation_all['description_key'])
display(pd.DataFrame({'training': train_sample['color'].value_counts(), 'validation': validation_all['color'].value_counts()}).reindex(LABELS))
print('Validation uses every eligible saved-validation review:', len(validation_all))

,training,validation
color,,
red,5,1562
white,5,1360


Validation uses every eligible saved-validation review: 2922


## 3. Compare a dummy reference and three TF-IDF classifiers

The same text representation, train rows, and validation rows are used for Naive Bayes, logistic regression, and SVM. The dummy model ignores text.

In [3]:
def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10_000, sublinear_tf=True)

models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Naive Bayes': make_pipeline(text_features(), MultinomialNB(alpha=1.0)),
    'Logistic regression': make_pipeline(text_features(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)),
    'Linear SVM': make_pipeline(text_features(), LinearSVC(C=1.0, max_iter=5000, random_state=SEED)),
}
X_train, y_train = train_sample['description'], train_sample['color']
X_val, y_val = validation_all['description'], validation_all['color']
score_rows = []
validation_predictions = {}

def record_scores(name, train_true, train_pred, val_true, val_pred):
    for split, actual, predicted in [('training', train_true, train_pred), ('validation', val_true, val_pred)]:
        score_rows.append({'model': name, 'split': split, 'n': len(actual), 'accuracy': accuracy_score(actual, predicted), 'macro_f1': f1_score(actual, predicted, labels=LABELS, average='macro', zero_division=0)})
    validation_predictions[name] = np.asarray(val_pred)

for name, model in models.items():
    model.fit(X_train, y_train)
    record_scores(name, y_train, model.predict(X_train), y_val, model.predict(X_val))
    print('Finished', name)

Finished Majority baseline
Finished Naive Bayes
Finished Logistic regression
Finished Linear SVM


In [4]:
def show_comparison():
    scores = pd.DataFrame(score_rows).pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
    scores.columns = [f'{metric}_{split}' for metric, split in scores.columns]
    scores['accuracy_gap'] = scores['accuracy_training'] - scores['accuracy_validation']
    scores['macro_f1_gap'] = scores['macro_f1_training'] - scores['macro_f1_validation']
    display(scores.sort_values('macro_f1_validation', ascending=False).round(3))
    for name, predicted in validation_predictions.items():
        print(f'\n{name}: validation report')
        print(classification_report(y_val, predicted, labels=LABELS, zero_division=0, digits=3))
        display(pd.DataFrame(confusion_matrix(y_val, predicted, labels=LABELS), index=['actual red', 'actual white'], columns=['predicted red', 'predicted white']))
    return scores

comparison = show_comparison()

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Logistic regression,1.0,0.786,1.000,0.783,0.214,0.217
Linear SVM,1.0,0.778,1.000,0.775,0.222,0.225
Naive Bayes,1.0,0.766,1.000,0.753,0.234,0.247
Majority baseline,0.5,0.535,0.333,0.348,-0.035,-0.015



Majority baseline: validation report
              precision    recall  f1-score   support

         red      0.535     1.000     0.697      1562
       white      0.000     0.000     0.000      1360

    accuracy                          0.535      2922
   macro avg      0.267     0.500     0.348      2922
weighted avg      0.286     0.535     0.372      2922



,predicted red,predicted white
actual red,1562,0
actual white,1360,0



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.716     0.931     0.809      1562
       white      0.879     0.576     0.696      1360

    accuracy                          0.766      2922
   macro avg      0.797     0.753     0.753      2922
weighted avg      0.792     0.766     0.756      2922



,predicted red,predicted white
actual red,1454,108
actual white,577,783



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.774     0.846     0.809      1562
       white      0.802     0.717     0.757      1360

    accuracy                          0.786      2922
   macro avg      0.788     0.782     0.783      2922
weighted avg      0.787     0.786     0.785      2922



,predicted red,predicted white
actual red,1322,240
actual white,385,975



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.766     0.843     0.803      1562
       white      0.796     0.704     0.747      1360

    accuracy                          0.778      2922
   macro avg      0.781     0.773     0.775      2922
weighted avg      0.780     0.778     0.777      2922



,predicted red,predicted white
actual red,1317,245
actual white,403,957


### Observed local TF-IDF results

These results were reproduced from the code above on 10 training reviews and **2,922** eligible validation reviews. DistilBERT has not been run here.

| Model | Training accuracy | Validation accuracy | Validation macro F1 |
|---|---:|---:|---:|
| Naive Bayes | 100.0% | 76.6% | 0.752 |
| Logistic regression | 100.0% | 78.6% | 0.783 |
| Linear SVM | 100.0% | 77.8% | 0.775 |

The majority baseline has 53.5% validation accuracy. The validation population contains 1,562 red and 1,360 white reviews. Model selection should use macro F1 and class-level errors as well as accuracy.

### Interpret each model's training/validation pattern

- **Naive Bayes:** If both scores are low, consider richer text features or tuning smoothing (`alpha`). If training is high but validation falls, add data or simplify the vocabulary.
- **Logistic regression:** If both scores are low, inspect the vocabulary and try a larger `C` only with a clear reason. If the training–validation gap is large, try a smaller `C` or more data. Its coefficients can explain red/white terms.
- **Linear SVM:** A large gap suggests trying stronger regularization (smaller `C`) or more data. Low scores on both splits suggest the current features or label scope need work. Its word weights are interpretable.
- **DistilBERT:** With a large gap, try more data or fewer epochs; with low scores on both splits, check labels and training setup. If it matches a simpler model, use the simpler model for this stage and revisit BERT when the scope grows. A randomly initialized classification head is expected.

Differences in training and validation scores are diagnostic clues, not formal estimates of statistical bias or variance. BERT uses pretrained language knowledge, so its 10- and 100-review runs answer a slightly different question than the sparse models.

## 4. Fine-tune DistilBERT on the same rows

A Colab GPU is recommended because backpropagation through DistilBERT is much more expensive than sparse TF-IDF fitting. Colab is for compute and setup convenience; it does not change the evaluation design. The setup cell below lists installed versions. If a package is missing or incompatible, uncomment its `%pip` line, run it once, **restart the kernel/runtime**, then rerun the notebook from the top. Do not upgrade packages and import them in the same live kernel.

In [ ]:
# Check the packages used by the DistilBERT section.
# If an import fails, uncomment and run the next line ONCE, then restart the kernel.
# %pip -q install -U "transformers==4.57.6" "accelerate>=1,<2" "datasets>=4,<5"
from importlib import metadata
for package in ('torch', 'transformers', 'accelerate', 'datasets', 'pyarrow'):
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: missing; install it and restart the kernel')

In [6]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

set_seed(SEED)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else ('Apple MPS' if torch.backends.mps.is_available() else 'CPU'))
MODEL_NAME = 'distilbert/distilbert-base-uncased'
label2id = {'red': 0, 'white': 1}
id2label = {v: k for k, v in label2id.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, frame):
        self.encodings = tokenizer(frame['description'].tolist(), truncation=True, max_length=128)
        self.labels = frame['color'].map(label2id).astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: torch.tensor(values[index]) for key, values in self.encodings.items()}, 'labels': torch.tensor(self.labels[index])}

bert_train = ReviewDataset(train_sample)
bert_val = ReviewDataset(validation_all)

/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: Apple MPS


In [7]:
def bert_metrics(result):
    predicted = result.predictions.argmax(axis=-1)
    return {'accuracy': accuracy_score(result.label_ids, predicted), 'macro_f1': f1_score(result.label_ids, predicted, labels=[0, 1], average='macro', zero_division=0)}

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
bert_args = TrainingArguments(
    output_dir=f'modeling_{MODEL_NUMBER}_bert_checkpoints',
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=1,
    fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
    report_to='none', seed=SEED,
)
bert_trainer = Trainer(
    model=bert_model, args=bert_args, train_dataset=bert_train, eval_dataset=bert_val,
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=bert_metrics,
)
bert_trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.691537,0.537303,0.495186
2,No log,0.690617,0.558864,0.519637
3,No log,0.690208,0.573580,0.548243


/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3, training_loss=0.6859793663024902, metrics={'train_runtime': 72.9938, 'train_samples_per_second': 0.411, 'train_steps_per_second': 0.041, 'total_flos': 582132123000.0, 'train_loss': 0.6859793663024902, 'epoch': 3.0})

In [8]:
train_output = bert_trainer.predict(bert_train)
val_output = bert_trainer.predict(bert_val)
bert_train_pred = np.array([id2label[int(i)] for i in train_output.predictions.argmax(axis=-1)])
bert_val_pred = np.array([id2label[int(i)] for i in val_output.predictions.argmax(axis=-1)])
record_scores('DistilBERT', y_train, bert_train_pred, y_val, bert_val_pred)
comparison = show_comparison()
print('Selected checkpoint:', bert_trainer.state.best_model_checkpoint)

/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Logistic regression,1.0,0.786,1.000,0.783,0.214,0.217
Linear SVM,1.0,0.778,1.000,0.775,0.222,0.225
Naive Bayes,1.0,0.766,1.000,0.753,0.234,0.247
DistilBERT,0.9,0.574,0.899,0.548,0.326,0.351
Majority baseline,0.5,0.535,0.333,0.348,-0.035,-0.015



Majority baseline: validation report
              precision    recall  f1-score   support

         red      0.535     1.000     0.697      1562
       white      0.000     0.000     0.000      1360

    accuracy                          0.535      2922
   macro avg      0.267     0.500     0.348      2922
weighted avg      0.286     0.535     0.372      2922



,predicted red,predicted white
actual red,1562,0
actual white,1360,0



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.716     0.931     0.809      1562
       white      0.879     0.576     0.696      1360

    accuracy                          0.766      2922
   macro avg      0.797     0.753     0.753      2922
weighted avg      0.792     0.766     0.756      2922



... (4 outputs left out)

,predicted red,predicted white
actual red,1317,245
actual white,403,957



DistilBERT: validation report
              precision    recall  f1-score   support

         red      0.577     0.758     0.655      1562
       white      0.566     0.362     0.441      1360

    accuracy                          0.574      2922
   macro avg      0.571     0.560     0.548      2922
weighted avg      0.572     0.574     0.556      2922



,predicted red,predicted white
actual red,1184,378
actual white,868,492


Selected checkpoint: modeling_1_bert_checkpoints/checkpoint-3


# Results/Next

each one is overfitting - just add more training data (obviously this is only for 10 obs)